In [3]:
from PIL import Image
import numpy as np

SRC = "precip.png"
INSET = 3  # px trimmed inside each border to remove the black line
THRESH = 240  # pixel counts as non white below this value
FRAC = 0.3  # a row or column belongs to a panel if over 30% of it is non white
NAMES = [["nov", "dec"], ["jan", "feb"]]

img = Image.open(SRC).convert("RGB")
nonwhite = (np.array(img) < THRESH).any(axis=2)


def runs(mask):
    """Return (start, end) pairs for each continuous stretch of True."""
    out, start = [], None
    for i, v in enumerate(mask):
        if v and start is None:
            start = i
        elif not v and start is not None:
            out.append((start, i))
            start = None
    if start is not None:
        out.append((start, len(mask)))
    return out


# Step 1: detect the panel boxes from the white gaps
boxes = {}
row_bands = runs(nonwhite.mean(axis=1) > FRAC)
assert len(row_bands) == 2, f"Expected 2 rows of panels, found {len(row_bands)}"

for r, (top, bottom) in enumerate(row_bands):
    col_bands = runs(nonwhite[top:bottom].mean(axis=0) > FRAC)
    assert len(col_bands) == 2, f"Expected 2 columns in row {r}, found {len(col_bands)}"
    for c, (left, right) in enumerate(col_bands):
        boxes[NAMES[r][c]] = (left + INSET, top + INSET, right - INSET, bottom - INSET)

# Step 2: the common size is the smallest panel in each direction
cw = min(r - l for l, t, r, b in boxes.values())
ch = min(b - t for l, t, r, b in boxes.values())
print(f"Common output size: {cw} x {ch}")

# Step 3: crop each panel to that size, centred so the trim is even
for name, (l, t, r, b) in boxes.items():
    dx = (r - l - cw) // 2
    dy = (b - t - ch) // 2
    panel = img.crop((l + dx, t + dy, l + dx + cw, t + dy + ch))
    panel.save(f"{name}_precip.png")  # PNG is lossless
    print(name, panel.size)

Common output size: 1028 x 1429
nov (1028, 1429)
dec (1028, 1429)
jan (1028, 1429)
feb (1028, 1429)


In [2]:
from pathlib import Path
from PIL import Image, ImageChops, ImageDraw, ImageFont

FOLDER = Path.cwd()  # the notebook is inside fig7, so the working folder is fig7
print(FOLDER)  # should end in ...images\fig7
MONTHS = {"d": "dec", "j": "jan", "f": "feb", "total": "winter"}
KINDS = ("precip", "temp")  # left panel first
LABELS = ("(a)", "(b)")

GAP_FRAC = 0.04  # gap between panels, fraction of panel width
STRIP_FRAC = 0.07  # label strip height, fraction of panel height
MARGIN_FRAC = 0.02  # outer margin, fraction of panel width
WHITE_TOL = 15  # pixel difference from white that counts as content


def load_panel(path):
    """Open a GIF or PNG, flatten transparency onto white, trim white margins."""
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    img = Image.open(path).convert("RGBA")
    img = Image.alpha_composite(Image.new("RGBA", img.size, "white"), img).convert(
        "RGB"
    )
    diff = ImageChops.difference(img, Image.new("RGB", img.size, "white")).convert("L")
    box = diff.point(lambda p: 255 if p > WHITE_TOL else 0).getbbox()
    return img.crop(box) if box else img


def fit(img, w, h):
    """Centre crop to the target aspect ratio, then resize only if needed."""
    target = w / h
    if img.width / img.height > target:  # too wide: trim the sides
        new_w = round(img.height * target)
        x = (img.width - new_w) // 2
        img = img.crop((x, 0, x + new_w, img.height))
    else:  # too tall: trim top and bottom
        new_h = round(img.width / target)
        y = (img.height - new_h) // 2
        img = img.crop((0, y, img.width, y + new_h))
    return img if img.size == (w, h) else img.resize((w, h), Image.LANCZOS)


def get_font(size):
    for name in ("arial.ttf", "Arial.ttf", "DejaVuSans.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    try:
        return ImageFont.load_default(size)  # needs Pillow 10.1 or newer
    except TypeError:
        return ImageFont.load_default()


# Load all six panels, then choose one common size (the smallest in each direction)
panels = {
    (k, m): load_panel(FOLDER / f"{k}_{MONTHS[m]}.gif") for m in MONTHS for k in KINDS
}

W = min(p.width for p in panels.values())
H = min(p.height for p in panels.values())
print(f"Panel size: {W} x {H}")

for (k, m), p in panels.items():
    if p.size != (W, H):
        print(f"Note: {k}_{MONTHS[m]} is {p.width} x {p.height} and will be adjusted")

gap = round(GAP_FRAC * W)
strip = round(STRIP_FRAC * H)
margin = round(MARGIN_FRAC * W)
font = get_font(round(strip * 0.6))

for m, month in MONTHS.items():
    canvas = Image.new("RGB", (2 * margin + 2 * W + gap, strip + H + margin), "white")
    draw = ImageDraw.Draw(canvas)

    for i, kind in enumerate(KINDS):
        x = margin + i * (W + gap)
        canvas.paste(fit(panels[(kind, m)], W, H), (x, strip))
        draw.text((x, strip // 2), LABELS[i], fill="black", font=font, anchor="lm")

    out = FOLDER / f"fig7_{m}.png"
    canvas.save(out)  # PNG keeps the colours exact
    print("Saved", out)

c:\CodingProjects\HTML_Projects\ComSci\bbc-news-clone\11538036-V1\images\fig7
Panel size: 531 x 753
Saved c:\CodingProjects\HTML_Projects\ComSci\bbc-news-clone\11538036-V1\images\fig7\fig7_d.png
Saved c:\CodingProjects\HTML_Projects\ComSci\bbc-news-clone\11538036-V1\images\fig7\fig7_j.png
Saved c:\CodingProjects\HTML_Projects\ComSci\bbc-news-clone\11538036-V1\images\fig7\fig7_f.png
Saved c:\CodingProjects\HTML_Projects\ComSci\bbc-news-clone\11538036-V1\images\fig7\fig7_total.png
